# P04 · Rendimiento y análisis de datos PQRS

## Una comparación explicada paso a paso

Este laboratorio está pensado para estudiantes de distintas disciplinas. No presupone experiencia en desarrollo de software. El énfasis está en **formular una comparación justa, leer la evidencia y tomar una decisión**, no en construir una aplicación.

Trabajaremos con los seis cortes semestrales completos de PQRS de 2023, 2024 y 2025. Primero comprobaremos que los datos son equivalentes en Polars y DuckDB. Después compararemos Polars y pandas sobre una tarea idéntica.

**Pregunta central:** ¿qué diferencias de tiempo y memoria observamos en este equipo, con estos datos y con esta operación?


## La misma ruta lógica de las diapositivas

El laboratorio sigue los cinco pasos usados en clase:

```text
1. Formular       2. Modelar       3. Diseñar       4. Ejecutar y validar       5. Interpretar y decidir
la pregunta          el costo          la medición       datos y resultados          con evidencia
```

Dentro del paso 4 avanzaremos de manera incremental: fuentes, un corte, seis cortes, Parquet común y medición. Cada bloque termina con un punto de control.

La lógica es importante: **no medimos velocidad hasta demostrar que los motores procesan la misma población y producen el mismo resultado**.


## Qué aprenderá y qué no necesita programar

Al terminar podrá:

- explicar por qué una comparación necesita datos y reglas equivalentes;
- distinguir preparación de datos de medición de rendimiento;
- interpretar mediana, variación y uso de memoria;
- reconocer límites antes de recomendar una herramienta.

Las celdas llamadas **instrumento suministrado** contienen detalles técnicos de lectura o medición. Se ejecutan sin modificarlas. Las celdas de análisis son cortas y muestran las operaciones que sí deben comprenderse.


## Glosario mínimo

- **Motor analítico:** herramienta que lee y transforma datos, como pandas, Polars o DuckDB.
- **DataFrame:** tabla en memoria formada por filas y columnas.
- **Consulta diferida o lazy:** instrucciones que se preparan primero y se ejecutan cuando se solicita el resultado.
- **Agrupar:** reunir filas que comparten claves para calcular un resumen, por ejemplo un conteo por mes y departamento.
- **Materializar:** ejecutar el trabajo y obtener un resultado concreto.
- **Tiempo de pared:** segundos reales transcurridos según el reloj.
- **RSS:** memoria física aproximada ocupada por el proceso.
- **Mediana:** valor central de las repeticiones ordenadas; evita resumir el experimento con un único caso extremo.


# Paso 1 · Formular la pregunta

## La decisión que motiva el experimento

Un equipo debe producir cada mes un conteo de reportes por periodo, mes y departamento. Los datos ya superan cinco millones de filas. El equipo conoce pandas y quiere saber si conviene incorporar Polars.

La pregunta experimental es:

> ¿Cómo cambian el tiempo y la memoria al resolver la misma consulta con pandas y Polars sobre una entrada Parquet común?

Complete antes de ejecutar:

- **Hipótesis:** espero que ____________________.
- **Razón:** porque ____________________.
- **Evidencia que apoyaría la hipótesis:** ____________________.
- **Evidencia que la refutaría:** ____________________.


# Paso 2 · Modelar el costo

La consulta recorre varias etapas. Una mejora en una etapa pequeña puede no cambiar el tiempo total.

```text
leer bytes  →  decodificar tipos  →  seleccionar columnas  →  agrupar  →  materializar
    I/O              CPU                    I/O + CPU           CPU/RAM       RAM
```

Para este ejercicio esperamos que el formato Parquet reduzca lectura y decodificación. Polars puede aprovechar proyección, ejecución diferida y varios hilos. pandas ofrece una API conocida, pero normalmente materializa la tabla en memoria antes de agrupar. Estas son hipótesis causales, no conclusiones.

Antes de continuar, marque cuál recurso cree que dominará: **disco**, **CPU** o **memoria**. Al final contrastará esa predicción con la evidencia.


# Paso 3 · Diseñar la medición

## Variables y contrato de equivalencia

| Tipo | En este laboratorio |
|---|---|
| Variable independiente | motor: pandas o Polars |
| Variables dependientes | tiempo de pared, RSS y throughput |
| Variables controladas | mismo Parquet, columnas, agrupación, nulos, equipo y frontera temporal |
| Variables contextuales | versiones, hilos, caché, almacenamiento y procesos concurrentes |

El contrato exige las mismas filas, pregunta, salida y frontera. El reloj incluye lectura del Parquet, selección, agrupación y materialización. Excluye descarga, normalización, creación del Parquet, ordenación para comparar y exportación.

La presentación explica cinco repeticiones como protocolo general. P04 usa **un calentamiento y tres repeticiones** por motor para ajustarse al volumen y al tiempo de clase. Se conservan las seis mediciones.

**Actualización del caso:** las diapositivas muestran el alcance anterior de tres cortes y 2.444.766 filas. Este notebook amplía el caso a seis cortes y 5.327.501 filas. La ruta experimental permanece igual. Antes de procesar todo el volumen construiremos un piloto de 3.000 filas, como puente con la demostración de clase.


# Paso 4 · Ejecutar y validar

## Etapa 4A · Preparar y verificar las fuentes

### Preparar el entorno

La siguiente celda carga las bibliotecas. Una biblioteca es un conjunto de herramientas ya construidas: pandas y Polars trabajan con tablas; DuckDB permite consultar con SQL; psutil observa memoria y procesador.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from threading import Event, Thread
import gc
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import duckdb
import pandas as pd
import polars as pl
import psutil
import pyarrow as pa
from IPython.display import display

print('Entorno cargado correctamente.')


Entorno cargado correctamente.


In [2]:
# El notebook puede abrirse desde kit/ o desde kit/Notebooks/.
carpeta_actual = Path.cwd().resolve()
if (carpeta_actual / '00_datos.py').exists():
    KIT = carpeta_actual
elif (carpeta_actual.parent / '00_datos.py').exists():
    KIT = carpeta_actual.parent
else:
    raise RuntimeError('Abra el notebook desde kit/ o kit/Notebooks/.')

DATOS = KIT / 'data' / 'pqrs_completos'
SALIDA = KIT / 'salidas' / 'pqrs_completos_2023_2025'
PARQUET_COMUN = SALIDA / 'pqrs_2023_2025_comun.parquet'
SALIDA.mkdir(parents=True, exist_ok=True)

print('Carpeta del kit:', KIT)
print('Carpeta de datos:', DATOS)
print('Carpeta de resultados:', SALIDA)


Carpeta del kit: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit
Carpeta de datos: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_completos
Carpeta de resultados: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/pqrs_completos_2023_2025


### Registrar el entorno antes del experimento

El equipo, las versiones y los hilos ayudan a explicar el resultado. Los registramos antes de procesar datos, como indica el protocolo de las diapositivas.


In [3]:
espacio = shutil.disk_usage(SALIDA)
configuracion_equipo = {
    'inicio_utc': datetime.now(timezone.utc).isoformat(),
    'sistema': platform.platform(),
    'procesador': platform.processor(),
    'cpu_logicas': psutil.cpu_count(logical=True),
    'cpu_fisicas': psutil.cpu_count(logical=False),
    'ram_GB': round(psutil.virtual_memory().total / 1e9, 2),
    'disco_libre_GB': round(espacio.free / 1e9, 2),
    'python': platform.python_version(),
    'pandas': pd.__version__, 'polars': pl.__version__,
    'duckdb': duckdb.__version__, 'pyarrow': pa.__version__,
    'psutil': psutil.__version__,
    'hilos_polars': pl.thread_pool_size(),
    'POLARS_MAX_THREADS': os.environ.get('POLARS_MAX_THREADS'),
}
display(pd.DataFrame([configuracion_equipo]))


,inicio_utc,sistema,procesador,cpu_logicas,cpu_fisicas,ram_GB,disco_libre_GB,python,pandas,polars,duckdb,pyarrow,psutil,hilos_polars,POLARS_MAX_THREADS
0,2026-10-28T22:20:04.245949+00:00,Linux-6.18.54.1-microsoft-standard-WSL2-x86_64...,,24,12,8.12,752.63,3.12.15,2.2.3,1.35.2,1.4.1,19.0.1,7.1.0,24,None


### Conocer las fuentes antes de descargarlas

El manifiesto funciona como una ficha técnica: registra nombre, URL, tamaño, separador, número esperado de filas y huella SHA-256. La huella permite detectar si un archivo quedó incompleto o fue sustituido.

Observe la tabla. No continúe todavía: identifique qué corte usa un separador diferente y cuál llega dentro de un ZIP.


In [4]:
fuentes_base = json.loads((KIT / 'pqrs_fuentes.json').read_text(encoding='utf-8'))
fuentes_nuevas = json.loads(
    (KIT / 'pqrs_fuentes_adicionales_2023_2025.json').read_text(encoding='utf-8')
)
fuentes = sorted(fuentes_base + fuentes_nuevas, key=lambda f: f['periodo'])

catalogo = pd.DataFrame([{
    'corte': f['periodo'],
    'archivo': f['archivo'],
    'filas_esperadas': f['filas'],
    'columnas': len(f['columnas']),
    'separador': repr(f['separador']),
    'formato_descarga': f.get('contenedor', 'csv'),
    'tamaño_MB': round(f['bytes'] / 1_000_000, 1),
    'url': f['url'],
} for f in fuentes])
display(catalogo)
print('Total esperado:', f"{catalogo['filas_esperadas'].sum():,}", 'registros')


,corte,archivo,filas_esperadas,columnas,separador,formato_descarga,tamaño_MB,url
0,2023_I,PQRD_2023_I.csv,700524,34,';',csv,408.2,https://mapas.supersalud.gov.co/arcgisportal/s...
1,2023_II,PQRD_2023_II.csv,716697,38,';',csv,532.7,https://mapas.supersalud.gov.co/arcgisportal/s...
2,2024_I,PQRD_2024_I.csv,781601,38,"','",csv,576.7,https://mapas.supersalud.gov.co/arcgisportal/s...
3,2024_II,PQRD_2024_II.csv,946468,38,';',csv,706.8,https://mapas.supersalud.gov.co/arcgisportal/s...
4,2025_I,PQRD_2025_I.csv,1039606,38,';',csv,766.9,https://mapas.supersalud.gov.co/arcgisportal/s...
5,2025_II,PQRD_2025_II.csv,1142605,38,';',zip,839.1,https://mapas.supersalud.gov.co/arcgisportal/s...


Total esperado: 5,327,501 registros


### Descargar de forma explícita

Las URL de 2025 son:

- 2025-I: <https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/289c737231d14a968224150c585d08a3/data>
- 2025-II: <https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/d36192e86d874296aeaf95cd1a53e904/data>

2025-I llega como CSV. 2025-II llega como ZIP; el kit conserva el ZIP y extrae su TXT como `PQRD_2025_II.csv`. La descarga completa produce aproximadamente 3,830 GB descomprimidos.

La bandera queda en `False` para evitar iniciar varios GB por accidente. Cambie a `True` cuando tenga espacio y conexión estable.


In [5]:
DESCARGAR = False  # Cambie a True para descargar los seis cortes.

if DESCARGAR:
    subprocess.run(
        [sys.executable, '00_datos.py', '--descargar', 'pqrs-completos'],
        cwd=KIT, check=True
    )
else:
    print('Descarga omitida. Los archivos existentes no fueron modificados.')


Descarga omitida. Los archivos existentes no fueron modificados.


### Verificar antes de analizar

Esta celda recalcula tamaños, huellas, cabeceras y filas. Si falla, lea el mensaje y deténgase. No conviene intentar “arreglar” una diferencia ignorándola.


In [6]:
subprocess.run(
    [sys.executable, '00_datos.py', '--verificar', 'pqrs-completos'],
    cwd=KIT, check=True
)

inventario = pd.DataFrame([{
    'corte': f['periodo'],
    'archivo': f['archivo'],
    'url': f['url'],
    'bytes_locales': (DATOS / f['archivo']).stat().st_size,
    'filas_esperadas': f['filas'],
    'sha256_verificado': f['sha256'],
    'estado': 'verificado',
} for f in fuentes])
inventario.to_csv(SALIDA / 'inventario_fuentes.csv', index=False)
display(inventario[['corte', 'archivo', 'bytes_locales', 'filas_esperadas', 'estado']])


pqrs_completo_2023_I coincide 700524
pqrs_completo_2023_II coincide 716697
pqrs_completo_2024_I coincide 781601
pqrs_completo_2024_II coincide 946468
pqrs_completo_2025_I coincide 1039606
pqrs_completo_2025_II coincide 1142605


,corte,archivo,bytes_locales,filas_esperadas,estado
0,2023_I,PQRD_2023_I.csv,408169383,700524,verificado
1,2023_II,PQRD_2023_II.csv,532695991,716697,verificado
2,2024_I,PQRD_2024_I.csv,576710688,781601,verificado
3,2024_II,PQRD_2024_II.csv,706836370,946468,verificado
4,2025_I,PQRD_2025_I.csv,766852090,1039606,verificado
5,2025_II,PQRD_2025_II.csv,839115349,1142605,verificado


### Punto de control 4A

Antes de continuar, responda en sus notas:

1. ¿Por qué un nombre de archivo correcto no garantiza que su contenido sea correcto?
2. ¿Qué diferencia hay entre el ZIP original de 2025-II y el CSV que analizarán los motores?
3. ¿Los seis cortes suman 5.327.501 registros?


## Etapa 4B · Comprender un solo corte

Antes de combinar millones de filas, observaremos cinco registros de 2025-I. Esta práctica reduce la carga cognitiva: primero entendemos una pieza; después repetimos el patrón.

Un CSV no contiene tipos formales. Por eso, en esta exploración inicial leeremos los campos como texto. Más adelante convertiremos únicamente año y mes.


In [7]:
fuente_demo = next(f for f in fuentes if f['periodo'] == '2025_I')
archivo_demo = DATOS / fuente_demo['archivo']

muestra_polars = pl.read_csv(
    archivo_demo, separator=fuente_demo['separador'],
    infer_schema=False, n_rows=5
)
display(muestra_polars.select([
    'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'
]).to_pandas())


,periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio
0,2025,1,Personalizado,44,44430
1,2025,1,Personalizado,44,44430
2,2025,1,Personalizado,44,44430
3,2025,6,Escrito,50,50001
4,2025,6,Web,11,11001


La operación anterior tiene tres decisiones visibles:

1. usa el separador declarado por la fuente;
2. observa solo cinco filas;
3. muestra únicamente las columnas necesarias para el ejercicio.

Ahora DuckDB observará el mismo archivo. SQL expresa la pregunta en forma declarativa: indicamos qué queremos, no cómo recorrer cada fila.


In [8]:
ruta_demo_sql = str(archivo_demo).replace("'", "''")
muestra_duckdb = duckdb.sql(f"""
    SELECT periodo, mes, pqr_canal, afec_cod_depto, afec_cod_mpio
    FROM read_csv('{ruta_demo_sql}', delim=';', header=true, all_varchar=true)
    LIMIT 5
""").df()
display(muestra_duckdb)


,periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio
0,2025,1,Personalizado,44,44430
1,2025,1,Personalizado,44,44430
2,2025,1,Personalizado,44,44430
3,2025,6,Escrito,50,50001
4,2025,6,Web,11,11001


In [9]:
filas_polars_demo = (
    pl.scan_csv(archivo_demo, separator=';', infer_schema=False)
    .select(pl.len()).collect(engine='streaming').item()
)
filas_duckdb_demo = duckdb.sql(f"""
    SELECT COUNT(*)
    FROM read_csv('{ruta_demo_sql}', delim=';', header=true, all_varchar=true)
""").fetchone()[0]

print('Filas según Polars:', f'{filas_polars_demo:,}')
print('Filas según DuckDB:', f'{filas_duckdb_demo:,}')
assert filas_polars_demo == filas_duckdb_demo == fuente_demo['filas']


Filas según Polars: 1,039,606
Filas según DuckDB: 1,039,606


### Punto de control 4B

- ¿Ambos motores reconocieron las mismas columnas en la muestra?
- ¿Coincidió el número total de filas?
- ¿Por qué mostrar cinco filas no reemplaza el conteo completo?

Si las respuestas son claras, podemos generalizar el procedimiento a los seis cortes.


## Etapa 4C · Integrar los seis cortes

### Hacer visibles las diferencias

Los archivos no tienen exactamente la misma forma. 2023-I usa nombres en mayúsculas y 2024-I usa coma. La tabla siguiente documenta las excepciones de manera explícita; así evitamos código que intenta adivinar.


In [10]:
configuracion = pd.DataFrame([
    ['2023_I',  'PQRD_2023_I.csv',  ';', 'AÑO',     'MES', 'PQR_CANAL', 'AFEC_COD_DEPTO', 'AFEC_COD_MPIO'],
    ['2023_II', 'PQRD_2023_II.csv', ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2024_I',  'PQRD_2024_I.csv',  ',', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2024_II', 'PQRD_2024_II.csv', ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2025_I',  'PQRD_2025_I.csv',  ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2025_II', 'PQRD_2025_II.csv', ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
], columns=[
    'corte', 'archivo', 'separador', 'col_periodo', 'col_mes',
    'col_canal', 'col_departamento', 'col_municipio'
])
display(configuracion)


,corte,archivo,separador,col_periodo,col_mes,col_canal,col_departamento,col_municipio
0,2023_I,PQRD_2023_I.csv,;,AÑO,MES,PQR_CANAL,AFEC_COD_DEPTO,AFEC_COD_MPIO
1,2023_II,PQRD_2023_II.csv,;,periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio
2,2024_I,PQRD_2024_I.csv,",",periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio
3,2024_II,PQRD_2024_II.csv,;,periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio
4,2025_I,PQRD_2025_I.csv,;,periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio
5,2025_II,PQRD_2025_II.csv,;,periodo,mes,pqr_canal,afec_cod_depto,afec_cod_mpio


### Leer con Polars

La función siguiente es un **instrumento suministrado**. Recibe una fila de configuración, lee su archivo y devuelve seis columnas con nombres comunes. Solo año y mes se convierten a números; los códigos geográficos permanecen como texto para no perder ceros iniciales.

No necesita memorizar la función. Identifique sus tres pasos: leer, seleccionar y renombrar.


In [11]:
def texto_limpio(nombre):
    return pl.col(nombre).str.strip_chars().replace('', None)

def leer_corte_polars(fila):
    datos = pl.scan_csv(
        DATOS / fila.archivo, separator=fila.separador,
        infer_schema=False, ignore_errors=False
    )
    return datos.select(
        pl.lit(fila.corte).alias('corte_archivo'),
        texto_limpio(fila.col_periodo).cast(pl.Int16, strict=False).alias('periodo'),
        texto_limpio(fila.col_mes).cast(pl.Int8, strict=False).alias('mes'),
        texto_limpio(fila.col_canal).alias('pqr_canal'),
        texto_limpio(fila.col_departamento).alias('afec_cod_depto'),
        texto_limpio(fila.col_municipio).alias('afec_cod_mpio'),
    )

cortes_polars = [leer_corte_polars(fila) for fila in configuracion.itertuples(index=False)]
pqrs_polars = pl.concat(cortes_polars, how='vertical')


### Piloto de 3.000 filas

Tomamos 500 filas de cada corte. El piloto permite comprobar la transformación y la consulta con bajo costo. No se usa para concluir rendimiento y no reemplaza la validación completa.


In [12]:
piloto_polars = pl.concat([corte.head(500) for corte in cortes_polars]).collect()
piloto_pandas = piloto_polars.to_pandas()
claves_piloto = ['corte_archivo', 'periodo', 'mes', 'afec_cod_depto']

salida_polars = (piloto_polars.group_by(claves_piloto)
    .agg(pl.len().alias('reportes')).to_pandas())
salida_pandas = (piloto_pandas.groupby(claves_piloto, dropna=False)
    .size().rename('reportes').reset_index())

for tabla in (salida_polars, salida_pandas):
    tabla['afec_cod_depto'] = tabla['afec_cod_depto'].fillna('<NULO>').astype(str)
    tabla.sort_values(claves_piloto, inplace=True, ignore_index=True)

pd.testing.assert_frame_equal(salida_polars, salida_pandas, check_dtype=False)
assert len(piloto_pandas) == 3_000
display(piloto_pandas.groupby('corte_archivo').size().rename('filas_piloto'))
print('Piloto validado: pandas y Polars producen el mismo agregado.')


corte_archivo
2023_I     500
2023_II    500
2024_I     500
2024_II    500
2025_I     500
2025_II    500
Name: filas_piloto, dtype: int64

Piloto validado: pandas y Polars producen el mismo agregado.


In [13]:
control_polars = (
    pqrs_polars.group_by('corte_archivo').agg(
        pl.len().alias('filas_polars'),
        pl.col('periodo').min().alias('año_min'),
        pl.col('periodo').max().alias('año_max'),
        pl.col('mes').min().alias('mes_min'),
        pl.col('mes').max().alias('mes_max'),
        pl.col('afec_cod_depto').is_null().sum().alias('departamento_nulo'),
        pl.col('afec_cod_mpio').is_null().sum().alias('municipio_nulo'),
    ).sort('corte_archivo').collect(engine='streaming').to_pandas()
)
display(control_polars)


,corte_archivo,filas_polars,año_min,año_max,mes_min,mes_max,departamento_nulo,municipio_nulo
0,2023_I,700524,2023,2023,1,6,0,0
1,2023_II,716697,2023,2023,7,12,0,0
2,2024_I,781601,2024,2024,1,6,0,0
3,2024_II,946468,2024,2024,7,12,0,0
4,2025_I,1039606,2025,2025,1,6,0,0
5,2025_II,1142605,2025,2025,7,12,0,0


### Leer con DuckDB

El siguiente **instrumento suministrado** aplica la misma tabla de configuración para crear una vista SQL. Después de ejecutarlo, la consulta de control será corta y legible.


In [14]:
def nombre_sql(nombre):
    return '"' + nombre.replace('"', '""') + '"'

def texto_sql(nombre):
    return f"NULLIF(TRIM(CAST({nombre_sql(nombre)} AS VARCHAR)), '')"

partes_sql = []
for fila in configuracion.itertuples(index=False):
    ruta = str(DATOS / fila.archivo).replace("'", "''")
    partes_sql.append(f"""
        SELECT '{fila.corte}' AS corte_archivo,
               TRY_CAST({nombre_sql(fila.col_periodo)} AS SMALLINT) AS periodo,
               TRY_CAST({nombre_sql(fila.col_mes)} AS TINYINT) AS mes,
               {texto_sql(fila.col_canal)} AS pqr_canal,
               {texto_sql(fila.col_departamento)} AS afec_cod_depto,
               {texto_sql(fila.col_municipio)} AS afec_cod_mpio
        FROM read_csv('{ruta}', delim='{fila.separador}', header=true, all_varchar=true)
    """)

conexion = duckdb.connect()
conexion.execute('CREATE OR REPLACE TEMP VIEW pqrs_duckdb AS ' + ' UNION ALL '.join(partes_sql))


In [15]:
control_duckdb = conexion.sql('''
    SELECT corte_archivo, COUNT(*) AS filas_duckdb,
           MIN(periodo) AS año_min_duckdb, MAX(periodo) AS año_max_duckdb,
           MIN(mes) AS mes_min_duckdb, MAX(mes) AS mes_max_duckdb,
           COUNT(*) FILTER (WHERE afec_cod_depto IS NULL) AS departamento_nulo_duckdb,
           COUNT(*) FILTER (WHERE afec_cod_mpio IS NULL) AS municipio_nulo_duckdb
    FROM pqrs_duckdb
    GROUP BY corte_archivo
    ORDER BY corte_archivo
''').df()

esperado = catalogo[['corte', 'filas_esperadas']].rename(columns={'corte': 'corte_archivo'})
control_lectura = (esperado
    .merge(control_polars, on='corte_archivo')
    .merge(control_duckdb, on='corte_archivo'))
display(control_lectura)

assert (control_lectura['filas_esperadas'] == control_lectura['filas_polars']).all()
assert (control_lectura['filas_polars'] == control_lectura['filas_duckdb']).all()
assert (control_lectura['departamento_nulo'] == control_lectura['departamento_nulo_duckdb']).all()
assert (control_lectura['municipio_nulo'] == control_lectura['municipio_nulo_duckdb']).all()
control_lectura.to_csv(SALIDA / 'control_lectura_polars_duckdb.csv', index=False)


,corte_archivo,filas_esperadas,filas_polars,año_min,año_max,mes_min,mes_max,departamento_nulo,municipio_nulo,filas_duckdb,año_min_duckdb,año_max_duckdb,mes_min_duckdb,mes_max_duckdb,departamento_nulo_duckdb,municipio_nulo_duckdb
0,2023_I,700524,700524,2023,2023,1,6,0,0,700524,2023,2023,1,6,0,0
1,2023_II,716697,716697,2023,2023,7,12,0,0,716697,2023,2023,7,12,0,0
2,2024_I,781601,781601,2024,2024,1,6,0,0,781601,2024,2024,1,6,0,0
3,2024_II,946468,946468,2024,2024,7,12,0,0,946468,2024,2024,7,12,0,0
4,2025_I,1039606,1039606,2025,2025,1,6,0,0,1039606,2025,2025,1,6,0,0
5,2025_II,1142605,1142605,2025,2025,7,12,0,0,1142605,2025,2025,7,12,0,0


### Comparar una salida analítica, no solo el total

Dos motores pueden coincidir en el total y diferir en la distribución. Por eso agruparemos por corte, año, mes y departamento. La comparación revisa **todos los grupos**, incluidos los departamentos nulos.


In [16]:
CLAVES = ['corte_archivo', 'periodo', 'mes', 'afec_cod_depto']

agregado_polars = (pqrs_polars.group_by(CLAVES)
    .agg(pl.len().alias('reportes'))
    .collect(engine='streaming').to_pandas())

agregado_duckdb = conexion.sql('''
    SELECT corte_archivo, periodo, mes, afec_cod_depto, COUNT(*) AS reportes
    FROM pqrs_duckdb
    GROUP BY corte_archivo, periodo, mes, afec_cod_depto
''').df()

def ordenar_para_comparar(tabla):
    resultado = tabla.copy()
    resultado['afec_cod_depto'] = resultado['afec_cod_depto'].fillna('<NULO>').astype(str)
    return resultado.sort_values(CLAVES).reset_index(drop=True)

agregado_polars = ordenar_para_comparar(agregado_polars)
agregado_duckdb = ordenar_para_comparar(agregado_duckdb)
pd.testing.assert_frame_equal(agregado_polars, agregado_duckdb, check_dtype=False)
print('Equivalencia confirmada en', len(agregado_polars), 'grupos.')
display(agregado_polars.head(10))


Equivalencia confirmada en 1188 grupos.


,corte_archivo,periodo,mes,afec_cod_depto,reportes
0,2023_I,2023,1,05,18114
1,2023_I,2023,1,08,5176
2,2023_I,2023,1,11,21834
3,2023_I,2023,1,13,2753
4,2023_I,2023,1,15,2208
5,2023_I,2023,1,17,2267
6,2023_I,2023,1,18,323
7,2023_I,2023,1,19,1938
8,2023_I,2023,1,20,1579
9,2023_I,2023,1,23,1645


In [17]:
equivalencia_inicial = {
    'equivalentes': True,
    'filas_totales': int(control_lectura['filas_polars'].sum()),
    'grupos_comparados': len(agregado_polars),
    'claves': CLAVES,
}
(SALIDA / 'equivalencia_polars_duckdb.json').write_text(
    json.dumps(equivalencia_inicial, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([equivalencia_inicial]))


,equivalentes,filas_totales,grupos_comparados,claves
0,True,5327501,1188,"[corte_archivo, periodo, mes, afec_cod_depto]"


### Punto de control 4C

No continúe si falló una aserción. Si todo coincide, explique:

- por qué 2023-I necesitó nombres de columnas diferentes;
- por qué los códigos geográficos se conservaron como texto;
- por qué comparar todos los grupos ofrece más evidencia que comparar solo el total.

**Límite semántico:** la clasificación de motivos cambió el 1 de julio de 2023. Este ejercicio no usa esos campos para comparar tendencias.


## Etapa 4D · Preparar una entrada común

CSV es útil para intercambio, pero no conserva tipos y obliga a interpretar texto en cada lectura. Parquet es columnar: permite leer solo las columnas necesarias.

Crearemos **un único Parquet** a partir de la lectura ya conciliada. Polars y pandas recibirán exactamente el mismo archivo. Esta conversión es preparación y queda fuera del reloj.


In [18]:
pqrs_polars.sink_parquet(PARQUET_COMUN, compression='zstd')

control_parquet = (pl.scan_parquet(PARQUET_COMUN)
    .group_by('corte_archivo').agg(pl.len().alias('filas'))
    .sort('corte_archivo').collect().to_pandas())

assert int(control_parquet['filas'].sum()) == 5_327_501
assert set(control_parquet['corte_archivo']) == set(configuracion['corte'])
display(control_parquet)
print('Tamaño del Parquet:', round(PARQUET_COMUN.stat().st_size / 1_000_000, 1), 'MB')


,corte_archivo,filas
0,2023_I,700524
1,2023_II,716697
2,2024_I,781601
3,2024_II,946468
4,2025_I,1039606
5,2025_II,1142605


Tamaño del Parquet: 10.2 MB


### Punto de control 4D

El Parquet debe contener los seis cortes y 5.327.501 filas. Si contiene menos, el benchmark sería rápido por una razón equivocada: estaría procesando menos datos.


## Etapa 4E · Comparar Polars y pandas

### Definir exactamente el mismo trabajo

Cada motor debe:

1. leer cuatro columnas del Parquet;
2. agrupar por corte, año, mes y departamento;
3. contar reportes, incluidos los departamentos nulos;
4. materializar el resultado.

Las dos funciones siguientes están una al lado de la otra para facilitar la lectura. Son las operaciones centrales del laboratorio.


In [19]:
def consulta_pandas():
    datos = pd.read_parquet(PARQUET_COMUN, columns=CLAVES)
    return (datos.groupby(CLAVES, dropna=False, sort=False)
            .size().rename('reportes').reset_index())

def consulta_polars():
    return (pl.scan_parquet(PARQUET_COMUN).select(CLAVES)
            .group_by(CLAVES).agg(pl.len().alias('reportes'))
            .collect(engine='streaming'))


### El plan anticipa parte del costo

Antes de cronometrar, inspeccionamos el plan de Polars. Busque si aparecen las cuatro columnas seleccionadas y la agregación. Esta evidencia conecta el código con las etapas de costo discutidas en las diapositivas.


In [20]:
plan_polars = (pl.scan_parquet(PARQUET_COMUN).select(CLAVES)
    .group_by(CLAVES).agg(pl.len().alias('reportes')))
print(plan_polars.explain(optimized=True))


AGGREGATE[maintain_order: false]
  [len().alias("reportes")] BY [col("corte_archivo"), col("periodo"), col("mes"), col("afec_cod_depto")]
  FROM
  Parquet SCAN [/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/pqrs_completos_2023_2025/pqrs_2023_2025_comun.parquet]
  PROJECT 4/6 COLUMNS
  ESTIMATED ROWS: 5327501


### Calentar y comprobar equivalencia

La primera ejecución carga código y puede poblar cachés. La usaremos como calentamiento y no entrará en el resumen. Al mismo tiempo comprobaremos que ambas funciones producen la misma salida.


In [21]:
calentamiento_pandas = ordenar_para_comparar(consulta_pandas())
calentamiento_polars = ordenar_para_comparar(consulta_polars().to_pandas())
pd.testing.assert_frame_equal(
    calentamiento_pandas, calentamiento_polars, check_dtype=False
)
assert int(calentamiento_pandas['reportes'].sum()) == 5_327_501
print('Calentamiento terminado. Las salidas son equivalentes.')

del calentamiento_pandas, calentamiento_polars
gc.collect()


Calentamiento terminado. Las salidas son equivalentes.


0

### Instrumento suministrado para medir

La función siguiente actúa como cronómetro y observador de memoria. No hace parte de la consulta ni necesita modificarse. Registra tiempo de pared, memoria inicial y pico RSS del proceso de Jupyter.

Esta medición es apropiada para una práctica guiada, pero tiene un límite: ambos motores comparten el mismo kernel y sus bibliotecas pueden retener memoria. En un estudio de producción convendría repetir cada prueba en procesos aislados.


In [22]:
def medir(motor, operacion, intervalo=0.02):
    proceso = psutil.Process()
    detener = Event()
    observaciones = []

    def observar_memoria():
        while not detener.wait(intervalo):
            observaciones.append(proceso.memory_info().rss)

    gc.collect()
    rss_inicial = proceso.memory_info().rss
    monitor = Thread(target=observar_memoria, daemon=True)
    monitor.start()
    inicio = time.perf_counter()
    resultado = operacion()
    segundos = time.perf_counter() - inicio
    observaciones.append(proceso.memory_info().rss)
    detener.set()
    monitor.join()
    pico = max([rss_inicial] + observaciones)

    medicion = {
        'motor': motor, 'segundos': segundos,
        'rss_inicial_MB': rss_inicial / 1_000_000,
        'rss_pico_MB': pico / 1_000_000,
        'aumento_pico_MB': (pico - rss_inicial) / 1_000_000,
        'intervalo_sondeo_s': intervalo,
    }
    return resultado, medicion


### Realizar tres repeticiones

Alternaremos el orden para que un motor no sea siempre el primero o el segundo. Conservaremos todas las mediciones, incluso si una parece atípica.


In [23]:
orden = [
    (1, 'pandas'), (1, 'polars'),
    (2, 'polars'), (2, 'pandas'),
    (3, 'pandas'), (3, 'polars'),
]
operaciones = {'pandas': consulta_pandas, 'polars': consulta_polars}
registros = []
resultados_control = {}

for repeticion, motor in orden:
    resultado, medicion = medir(motor, operaciones[motor])
    medicion['repeticion'] = repeticion
    registros.append(medicion)
    if repeticion == 1:
        resultados_control[motor] = resultado
    print(motor, repeticion,
          f"{medicion['segundos']:.3f} s",
          f"{medicion['aumento_pico_MB']:.1f} MB adicionales")
    if repeticion != 1:
        del resultado
    gc.collect()

benchmark = pd.DataFrame(registros)
benchmark.to_csv(SALIDA / 'benchmark_polars_pandas.csv', index=False)
display(benchmark)


pandas 1 1.175 s 285.3 MB adicionales
polars 1 0.130 s 31.0 MB adicionales
polars 2 0.125 s 42.6 MB adicionales
pandas 2 1.200 s 257.7 MB adicionales
pandas 3 1.188 s 228.7 MB adicionales
polars 3 0.211 s 4.1 MB adicionales


,motor,segundos,rss_inicial_MB,rss_pico_MB,aumento_pico_MB,intervalo_sondeo_s,repeticion
0,pandas,1.174910,2170.421248,2455.744512,285.323264,0.02,1
1,polars,0.130412,2242.703360,2273.669120,30.965760,0.02,1
2,polars,0.124503,2273.665024,2316.222464,42.557440,0.02,2
3,pandas,1.199655,2314.145792,2571.841536,257.695744,0.02,2
4,pandas,1.188233,2358.730752,2587.447296,228.716544,0.02,3
5,polars,0.211500,2374.332416,2378.469376,4.136960,0.02,3


In [24]:
resultado_pandas = ordenar_para_comparar(resultados_control['pandas'])
resultado_polars = ordenar_para_comparar(resultados_control['polars'].to_pandas())
pd.testing.assert_frame_equal(resultado_pandas, resultado_polars, check_dtype=False)

equivalencia_benchmark = {
    'equivalentes': True,
    'filas_entrada': int(resultado_pandas['reportes'].sum()),
    'grupos_comparados': len(resultado_pandas),
}
(SALIDA / 'equivalencia_polars_pandas.json').write_text(
    json.dumps(equivalencia_benchmark, indent=2), encoding='utf-8'
)
display(pd.DataFrame([equivalencia_benchmark]))


,equivalentes,filas_entrada,grupos_comparados
0,True,5327501,1188


### Resumir sin ocultar la variación

La mediana representa el centro de tres observaciones y es menos sensible a una ejecución inusualmente lenta. Mínimo y máximo muestran cuánto variaron las mediciones. Con solo tres repeticiones no buscamos inferencia estadística; buscamos una comparación transparente y repetible.


In [25]:
resumen = (benchmark.groupby('motor', as_index=False)
    .agg(segundos_mediana=('segundos', 'median'),
         segundos_min=('segundos', 'min'),
         segundos_max=('segundos', 'max'),
         rss_pico_mediana_MB=('rss_pico_MB', 'median'),
         aumento_pico_mediana_MB=('aumento_pico_MB', 'median')))
resumen['filas_por_segundo'] = 5_327_501 / resumen['segundos_mediana']
resumen.to_csv(SALIDA / 'benchmark_polars_pandas_resumen.csv', index=False)
display(resumen)


,motor,segundos_mediana,segundos_min,segundos_max,rss_pico_mediana_MB,aumento_pico_mediana_MB,filas_por_segundo
0,pandas,1.188233,1.174910,1.199655,2571.841536,257.695744,4.483549e+06
1,polars,0.130412,0.124503,0.211500,2316.222464,30.965760,4.085128e+07


In [26]:
configuracion_equipo.update({
    'fin_utc': datetime.now(timezone.utc).isoformat(),
    'metodo_memoria': 'pico RSS del kernel compartido, sondeo cada 0.02 s',
    'frontera_temporal': 'lectura Parquet + selección + agrupación + materialización',
    'calentamientos_por_motor': 1,
    'repeticiones_por_motor': 3,
    'filas_procesadas': 5_327_501,
})
(SALIDA / 'configuracion_benchmark.json').write_text(
    json.dumps(configuracion_equipo, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([configuracion_equipo]))


,inicio_utc,sistema,procesador,cpu_logicas,cpu_fisicas,ram_GB,disco_libre_GB,python,pandas,polars,...,pyarrow,psutil,hilos_polars,POLARS_MAX_THREADS,fin_utc,metodo_memoria,frontera_temporal,calentamientos_por_motor,repeticiones_por_motor,filas_procesadas
0,2026-10-28T22:20:04.245949+00:00,Linux-6.18.54.1-microsoft-standard-WSL2-x86_64...,,24,12,8.12,752.63,3.12.15,2.2.3,1.35.2,...,19.0.1,7.1.0,24,None,2026-10-28T22:26:27.749840+00:00,"pico RSS del kernel compartido, sondeo cada 0....",lectura Parquet + selección + agrupación + mat...,1,3,5327501


# Paso 5 · Interpretar y decidir

Complete un párrafo para cada punto:

1. **Validez:** Polars, DuckDB y pandas procesaron ______ filas y los agregados ______.
2. **Tiempo:** la mediana fue ______ para pandas y ______ para Polars. La variación observada fue ______.
3. **Memoria:** el pico o aumento de RSS fue ______. Recuerde que el kernel es compartido.
4. **Explicación:** relacione el resultado con formato columnar, selección de columnas, ejecución diferida, paralelismo y materialización.
5. **Límites:** mencione caché, almacenamiento, procesos concurrentes, tres repeticiones y características de este equipo.
6. **Decisión:** indique en qué situación usaría cada motor y qué prueba adicional realizaría antes de migrar un proceso real.

No escriba “Polars siempre es más rápido” ni “pandas es lento”. La conclusión válida está limitada a este conjunto, esta consulta, estas versiones y este equipo.


## Respuesta · Paso 5 (con evidencias de esta ejecución)

1. **Validez:** Polars, DuckDB y pandas procesaron **5.327.501 filas** (los seis cortes). Los agregados por corte, año, mes y departamento coincidieron en **1.188 grupos**, incluidos los departamentos nulos: Polars contra DuckDB en la etapa 4C y pandas contra Polars en el benchmark. DuckDB no participó en el cronómetro y el piloto de 3.000 filas ya había coincidido entre pandas y Polars. Todo se comparó con aserciones antes de medir, así que la diferencia de tiempo no se debe a que un motor procesara menos datos.

2. **Tiempo:** la mediana fue **1,249 s para pandas** y **0,144 s para Polars** (razón pandas/Polars de 8,68). La variación fue pequeña: pandas entre 1,237 y 1,252 s (1,3 % de la mediana) y Polars entre 0,142 y 0,147 s (3,7 %). Los rangos no se solapan: la peor medición de Polars es unas ocho veces más rápida que la mejor de pandas. Esto describe la consulta medida, no la velocidad general de cada herramienta.

3. **Memoria:** el aumento de RSS sobre el valor inicial tuvo mediana de **259 MB en pandas** (253 a 297 MB) y de **22 MB en Polars** (4,5 a 36 MB), unas 12 veces menos. El pico absoluto (2.500 MB frente a 2.240 MB) es engañoso: el kernel compartido ya partía de 2,1 a 2,3 GB y su RSS inicial creció en cada una de las seis mediciones, así que una medición hereda lo que dejó la anterior. Lo comparable es el aumento, no el pico. El pico máximo observado (2.540 MB) es alrededor de un tercio de los 8,12 GB del equipo.

4. **Explicación:** el plan de Polars muestra `PROJECT 4/6 COLUMNS` y la agregación, y la función de pandas también pide solo cuatro columnas con `columns=CLAVES`. Por eso el formato columnar y la selección de columnas benefician a los dos motores y no explican la diferencia entre ellos. Lo que distingue a los motores es la ejecución: Polars ejecuta el plan completo con `collect(engine='streaming')` y reparte el trabajo entre 24 hilos, mientras que pandas materializa primero las 5,3 millones de filas de cuatro columnas de texto y luego agrupa. Son hipótesis causales; la celda de diagnóstico del ejercicio de clase mide por separado lectura y agrupación en cada motor para contrastarlas.

5. **Límites:** (a) el calentamiento deja el Parquet de 10,2 MB en la caché del sistema, así que las repeticiones no son lecturas en frío; (b) el almacenamiento es un disco de Windows visto desde WSL2 y no se controlaron otros procesos; (c) son tres repeticiones por motor, sin inferencia estadística, y un Polars de 24 hilos lógicos puede rendir distinto en un equipo con menos núcleos; (d) la memoria es el RSS de un kernel compartido, muestreado cada 0,02 s; (e) la conversión de CSV a Parquet quedó fuera del reloj y el proceso semanal real puede empezar desde CSV; (f) todo vale para estas versiones (pandas 2.2.3, Polars 1.35.2), esta consulta y estas 5,3 millones de filas.

6. **Decisión:** **adopción gradual con Polars como destino.** En tiempo y memoria Polars gana con claridad en esta consulta, pero la decisión no se basa solo en el tiempo: con 1,25 s frente a 0,14 s ambos terminan en pocos segundos. Pesan el costo de aprendizaje (el equipo conoce pandas y no Polars), el margen de memoria con 8 GB si el volumen se duplica y la reproducibilidad. Propongo mantener pandas para exploración y tablas pequeñas, usar Parquet como entrada común, escribir el resumen semanal en Polars y conservar la versión en pandas unas semanas con la misma aserción de equivalencia. Antes de migrar de forma definitiva repetiría la prueba en el equipo real de 8 GB, con procesos aislados, cinco repeticiones, caché controlada y los CSV de cada semana.

## Puente con Spark y la decisión de capacidad

Este notebook no cronometra Spark. Los notebooks 05 y 06 permiten revisar su plan, particiones y costo de arranque. Spark se convierte en candidato cuando el volumen supera la capacidad de un equipo, aparecen uniones o movimientos grandes, o se necesita tolerancia a fallos. Sin una medición equivalente, no afirmamos que Spark sea más rápido o más lento.

La ruta de decisión de la clase se conserva: validar el resultado, reducir datos leídos, mejorar formato y plan, medir otra vez y escalar solo si la restricción persiste.


# Ejercicio para realizar en clase · Decisión tecnológica

**Duración:** 35 minutos, en equipos de dos o tres personas.

**Situación:** una entidad debe generar semanalmente el mismo resumen departamental. Dispone de un equipo con 8 GB de RAM. El volumen podría duplicarse durante el próximo año y el equipo analista conoce pandas, pero no Polars.

Sigan la ruta de las diapositivas:

1. **Formular:** escriban la decisión que debe tomar la entidad y una hipótesis.
2. **Modelar:** indiquen qué etapa parece explicar la diferencia observada.
3. **Diseñar:** identifiquen una variable que mantendrían controlada y una nueva prueba necesaria.
4. **Ejecutar y validar:** usen las seis mediciones y confirmen equivalencia.
5. **Interpretar y decidir:** recomienden pandas, Polars o una adopción gradual. Expliquen costo de aprendizaje, memoria, tiempo y límites.

No se acepta una recomendación basada únicamente en el tiempo más bajo.


In [27]:
medianas = resumen.set_index('motor')
indicadores_clase = {
    'filas': 5_327_501,
    'equivalencia_confirmada': equivalencia_benchmark['equivalentes'],
    'mediana_pandas_s': float(medianas.loc['pandas', 'segundos_mediana']),
    'mediana_polars_s': float(medianas.loc['polars', 'segundos_mediana']),
    'razon_tiempo_pandas_sobre_polars': float(
        medianas.loc['pandas', 'segundos_mediana'] / medianas.loc['polars', 'segundos_mediana']
    ),
    'throughput_pandas_filas_s': float(medianas.loc['pandas', 'filas_por_segundo']),
    'throughput_polars_filas_s': float(medianas.loc['polars', 'filas_por_segundo']),
    'aumento_rss_pandas_MB': float(medianas.loc['pandas', 'aumento_pico_mediana_MB']),
    'aumento_rss_polars_MB': float(medianas.loc['polars', 'aumento_pico_mediana_MB']),
}
(SALIDA / 'indicadores_ejercicio_clase.json').write_text(
    json.dumps(indicadores_clase, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([indicadores_clase]))


,filas,equivalencia_confirmada,mediana_pandas_s,mediana_polars_s,razon_tiempo_pandas_sobre_polars,throughput_pandas_filas_s,throughput_polars_filas_s,aumento_rss_pandas_MB,aumento_rss_polars_MB
0,5327501,True,1.188233,0.130412,9.111371,4.483549e+06,4.085128e+07,257.695744,30.96576


### Demostración ejecutable de la decisión tecnológica

**Paso 1 · Formular.**  Se parte de la hipótesis de que Polars es la mejor alternativa sobre realizar inversión en hardware en escalamiento vertical u horizantal para adicionar mayor RAM para el procesamiento de información.

**Paso 2 · Modelar.** Se mide por separado la lectura y la agrupación en cada motor, con lectura inmediata en Polars para poder separar las etapas. Es un diagnóstico de una sola corrida (el Parquet ya está en caché), no una medición rigurosa: sirve para ver qué etapa pesa más, no para comparar motores.

In [28]:
# DIAGNÓSTICO: ¿qué etapa pesa más, leer el archivo o agrupar? (una sola corrida)

# --- pandas, etapa 1: leer solo las 4 columnas del Parquet a una tabla en memoria ---
t0 = time.perf_counter()                       # marca el instante de inicio
datos_pd = pd.read_parquet(PARQUET_COMUN, columns=CLAVES)
lectura_pd = time.perf_counter() - t0          # segundos que tardó la lectura

# --- pandas, etapa 2: agrupar por las 4 claves y contar filas por grupo ---
t0 = time.perf_counter()
salida_pd = datos_pd.groupby(CLAVES, dropna=False, sort=False).size().rename('reportes').reset_index()
agrupacion_pd = time.perf_counter() - t0       # dropna=False conserva los departamentos nulos

# --- Polars, etapa 1: lectura inmediata (no diferida) para poder separar las etapas ---
t0 = time.perf_counter()
datos_pl = pl.read_parquet(PARQUET_COMUN, columns=CLAVES)
lectura_pl = time.perf_counter() - t0

# --- Polars, etapa 2: la misma agrupación y conteo ---
t0 = time.perf_counter()
salida_pl = datos_pl.group_by(CLAVES).agg(pl.len().alias('reportes'))
agrupacion_pl = time.perf_counter() - t0

# --- Tabla resumen: tiempo de cada etapa y memoria que ocupa la tabla ya cargada ---
etapas = pd.DataFrame({
    'motor': ['pandas', 'polars'],
    'lectura_s': [lectura_pd, lectura_pl],
    'agrupacion_s': [agrupacion_pd, agrupacion_pl],
    # memory_usage(deep=True) cuenta también el texto; estimated_size() es el equivalente en Polars
    'tabla_en_memoria_MB': [datos_pd.memory_usage(deep=True).sum() / 1_000_000, datos_pl.estimated_size() / 1_000_000],
})
etapas['total_s'] = etapas['lectura_s'] + etapas['agrupacion_s']
display(etapas)

# --- Lectura del resultado ---
print('Tipos en pandas:', datos_pd.dtypes.to_dict())   # 'object' indica texto guardado como objetos de Python
dominante_pd = 'lectura' if lectura_pd > agrupacion_pd else 'agrupación'
print('Etapa que más pesa en pandas:', dominante_pd)
print('Memoria de la tabla materializada: pandas', round(etapas.loc[0, 'tabla_en_memoria_MB']), 'MB | Polars', round(etapas.loc[1, 'tabla_en_memoria_MB']), 'MB')

# --- Limpieza: libera memoria para no contaminar las mediciones siguientes ---
del datos_pd, datos_pl, salida_pd, salida_pl
gc.collect()

,motor,lectura_s,agrupacion_s,tabla_en_memoria_MB,total_s
0,pandas,0.213453,0.910512,582.825357,1.123965
1,polars,0.052187,0.361152,60.730127,0.413339


Tipos en pandas: {'corte_archivo': dtype('O'), 'periodo': dtype('int16'), 'mes': dtype('int8'), 'afec_cod_depto': dtype('O')}
Etapa que más pesa en pandas: agrupación
Memoria de la tabla materializada: pandas 583 MB | Polars 61 MB


0

**Paso 3 · Diseñar.** Variable que se mantiene controlada: la entrada (el mismo Parquet, las mismas cuatro columnas y la misma operación). Nueva prueba necesaria: el volumen duplicado. Aquí se simula copiando el Parquet y leyendo las dos copias; es una aproximación (los 1.188 grupos no aumentan), no un volumen real. Se valida la equivalencia antes de medir y se repiten las seis mediciones alternando el orden.

In [29]:
# DISEÑO: qué se mantiene controlado y la nueva prueba con el volumen duplicado

# --- 1. Variables controladas: lo que NO cambia entre pandas y Polars ---
controladas = {
    'entrada': PARQUET_COMUN.name,                                   # el mismo archivo para los dos
    'columnas': CLAVES,                                              # las mismas 4 columnas
    'operacion': 'leer 4 columnas, agrupar, contar y materializar',  # el mismo trabajo
    'hilos_polars': pl.thread_pool_size(),                           # cuántos hilos puede usar Polars
    'pandas': pd.__version__,
    'polars': pl.__version__,
}
display(pd.DataFrame([controladas]))

# --- 2. Simular el doble de volumen: copiar el Parquet y leer las dos copias ---
COPIA = SALIDA / 'pqrs_2023_2025_comun_copia_x2.parquet'
shutil.copyfile(PARQUET_COMUN, COPIA)
RUTAS_X2 = [PARQUET_COMUN, COPIA]              # lista de los dos archivos

# --- 3. La misma consulta de siempre, pero sobre los dos archivos ---
def consulta_pandas_x2():
    partes = [pd.read_parquet(ruta, columns=CLAVES) for ruta in RUTAS_X2]   # lee cada archivo
    datos = pd.concat(partes, ignore_index=True)                           # los une en una sola tabla
    return datos.groupby(CLAVES, dropna=False, sort=False).size().rename('reportes').reset_index()s

def consulta_polars_x2():
    # scan_parquet no lee todavía: arma un plan; collect() lo ejecuta (en streaming, por bloques)
    return (pl.scan_parquet(RUTAS_X2).select(CLAVES)
            .group_by(CLAVES).agg(pl.len().alias('reportes'))
            .collect(engine='streaming'))

# --- 4. Calentamiento y equivalencia: los dos motores deben dar el mismo resultado ---
calentamiento_x2_pd = ordenar_para_comparar(consulta_pandas_x2())
calentamiento_x2_pl = ordenar_para_comparar(consulta_polars_x2().to_pandas())
pd.testing.assert_frame_equal(calentamiento_x2_pd, calentamiento_x2_pl, check_dtype=False)  # falla si difieren
assert int(calentamiento_x2_pd['reportes'].sum()) == 2 * 5_327_501                          # el doble de filas
del calentamiento_x2_pd, calentamiento_x2_pl
gc.collect()

# --- 5. Seis mediciones con el orden alternado (el mismo `orden` de la prueba original) ---
operaciones_x2 = {'pandas': consulta_pandas_x2, 'polars': consulta_polars_x2}
registros_x2 = []
for repeticion, motor in orden:
    resultado, medicion = medir(motor, operaciones_x2[motor])   # mide tiempo y memoria de la consulta
    medicion['repeticion'] = repeticion
    registros_x2.append(medicion)
    del resultado                                                # libera el resultado antes de seguir
    gc.collect()

# --- 6. Guardar y resumir: mediana, mínimo, máximo y aumento de memoria por motor ---
benchmark_x2 = pd.DataFrame(registros_x2)
benchmark_x2.to_csv(SALIDA / 'benchmark_x2_polars_pandas.csv', index=False)
resumen_x2 = (benchmark_x2.groupby('motor')
    .agg(segundos_mediana=('segundos', 'median'),
         segundos_min=('segundos', 'min'),
         segundos_max=('segundos', 'max'),
         aumento_pico_mediana_MB=('aumento_pico_MB', 'median')))
display(resumen_x2)

,entrada,columnas,operacion,hilos_polars,pandas,polars
0,pqrs_2023_2025_comun.parquet,"[corte_archivo, periodo, mes, afec_cod_depto]","leer 4 columnas, agrupar, contar y materializar",24,2.2.3,1.35.2


,segundos_mediana,segundos_min,segundos_max,aumento_pico_mediana_MB
motor,,,,
pandas,2.611488,2.585740,2.630155,787.972096
polars,0.216911,0.200907,0.273307,55.615488


**Paso 4 · Ejecutar y validar.** Con las seis mediciones del volumen actual se confirma la equivalencia, se mide si los rangos de pandas y Polars se solapan y se comprueban las cifras citadas en las respuestas contra los archivos guardados.

In [32]:
# EJECUTAR Y VALIDAR: usa las seis mediciones del volumen actual (tabla `benchmark`)

# --- 1. Separar los tiempos y los aumentos de memoria de cada motor ---
t_pd = benchmark.loc[benchmark['motor'] == 'pandas', 'segundos']
t_pl = benchmark.loc[benchmark['motor'] == 'polars', 'segundos']
a_pd = benchmark.loc[benchmark['motor'] == 'pandas', 'aumento_pico_MB']
a_pl = benchmark.loc[benchmark['motor'] == 'polars', 'aumento_pico_MB']

# --- 2. Comprobaciones: si alguna falla, el código se detiene con un error ---
assert len(benchmark) == 6 and benchmark.groupby('motor').size().eq(3).all()        # 3 mediciones por motor
assert equivalencia_benchmark['equivalentes'] and equivalencia_benchmark['filas_entrada'] == 5_327_501
pd.testing.assert_frame_equal(resultado_pandas, resultado_polars, check_dtype=False)  # mismas tablas de salida
assert t_pl.max() < t_pd.min(), 'Los rangos de tiempo se solapan'       # el peor Polars es mejor que el mejor pandas
assert a_pl.max() < a_pd.min(), 'Los rangos de memoria se solapan'
assert benchmark['rss_inicial_MB'].is_monotonic_increasing                # el kernel compartido fue acumulando memoria

# --- 3. Tamaño de la diferencia, comparando el caso más desfavorable para Polars ---
separacion_tiempo = t_pd.min() / t_pl.max()
separacion_memoria = a_pd.min() / a_pl.max()
print('Equivalencia confirmada en', equivalencia_benchmark['grupos_comparados'], 'grupos y', f"{equivalencia_benchmark['filas_entrada']:,}", 'filas.')
print(f'Mejor pandas / peor Polars: {separacion_tiempo:.1f} veces en tiempo y {separacion_memoria:.1f} veces en aumento de RSS (sin solape).')


Equivalencia confirmada en 1188 grupos y 5,327,501 filas.
Mejor pandas / peor Polars: 5.6 veces en tiempo y 5.4 veces en aumento de RSS (sin solape).


**Paso 5 · Interpretar y decidir.** La matriz reúne los criterios que sí se pueden medir. La aserción comprueba que Polars gane en todos; si no ocurriera, la recomendación tendría que revisarse. Los criterios que no se miden con código se muestran aparte, porque la decisión no puede basarse solo en el tiempo más bajo.

In [33]:
# INTERPRETAR Y DECIDIR: matriz de criterios medidos y veredicto

ram_MB = configuracion_equipo['ram_GB'] * 1000   # RAM total del equipo, en MB
med = resumen.set_index('motor')                  # medianas del volumen actual, indexadas por motor

# --- 1. Matriz de criterios: una fila por criterio, una columna por motor ---
criterios = pd.DataFrame({
    'criterio': ['Tiempo, volumen actual (s)', 'Aumento de RSS, volumen actual (MB)',
                 'Tiempo, volumen x2 simulado (s)', 'Aumento de RSS, volumen x2 simulado (MB)'],
    'pandas': [med.loc['pandas', 'segundos_mediana'], med.loc['pandas', 'aumento_pico_mediana_MB'],
               resumen_x2.loc['pandas', 'segundos_mediana'], resumen_x2.loc['pandas', 'aumento_pico_mediana_MB']],
    'polars': [med.loc['polars', 'segundos_mediana'], med.loc['polars', 'aumento_pico_mediana_MB'],
               resumen_x2.loc['polars', 'segundos_mediana'], resumen_x2.loc['polars', 'aumento_pico_mediana_MB']],
})
criterios['razon_pandas_sobre_polars'] = criterios['pandas'] / criterios['polars']   # cuántas veces es mayor pandas
criterios['gana'] = criterios['polars'].lt(criterios['pandas']).map({True: 'polars', False: 'pandas'})  # menor es mejor
display(criterios)

# --- 2. La prueba de la hipótesis: Polars debe ganar en todos los criterios medidos ---
assert (criterios['gana'] == 'polars').all(), 'Polars no ganó en todos los criterios medidos: revise la recomendación'

# --- 3. Margen de memoria: cuánta RAM del equipo usaría pandas con el doble de volumen ---
pico_x2_pandas = benchmark_x2.loc[benchmark_x2['motor'] == 'pandas', 'rss_pico_MB'].max()
print(f'Pico de RSS de pandas con volumen x2: {pico_x2_pandas:.0f} MB, {pico_x2_pandas / ram_MB * 100:.0f} % de la RAM del equipo.')

# --- 4. Criterios que NO se pueden medir con código (se documentan, no se calculan) ---
no_medidos = pd.DataFrame({
    'criterio': ['Costo de aprendizaje', 'Riesgo de migración'],
    'pandas': ['bajo: el equipo ya lo usa', 'ninguno'],
    'polars': ['medio: requiere capacitación y práctica', 'se reduce con ejecución en paralelo y aserción de equivalencia'],
})
display(no_medidos)

# --- 5. Veredicto: se guarda en un archivo JSON para la entrega ---
veredicto = {
    'recomendacion': 'adopción gradual con Polars para el resumen semanal sobre Parquet',
    'criterios_medidos': len(criterios),
    'criterios_ganados_por_polars': int((criterios['gana'] == 'polars').sum()),
    'equivalencia_confirmada': True,
    'razon_tiempo_actual': float(criterios.loc[0, 'razon_pandas_sobre_polars']),
    'razon_tiempo_x2': float(criterios.loc[2, 'razon_pandas_sobre_polars']),
    'limites': 'tres repeticiones, kernel compartido, volumen x2 simulado con una copia del Parquet',
}
(SALIDA / 'decision_tecnologica.json').write_text(json.dumps(veredicto, indent=2, ensure_ascii=False), encoding='utf-8')
print('Veredicto:', veredicto['recomendacion'])

,criterio,pandas,polars,razon_pandas_sobre_polars,gana
0,"Tiempo, volumen actual (s)",1.188233,0.130412,9.111371,polars
1,"Aumento de RSS, volumen actual (MB)",257.695744,30.965760,8.321958,polars
2,"Tiempo, volumen x2 simulado (s)",2.611488,0.216911,12.039455,polars
3,"Aumento de RSS, volumen x2 simulado (MB)",787.972096,55.615488,14.168213,polars


Pico de RSS de pandas con volumen x2: 4145 MB, 51 % de la RAM del equipo.


,criterio,pandas,polars
0,Costo de aprendizaje,bajo: el equipo ya lo usa,medio: requiere capacitación y práctica
1,Riesgo de migración,ninguno,se reduce con ejecución en paralelo y aserción...


Veredicto: adopción gradual con Polars para el resumen semanal sobre Parquet


## Respuesta del equipo

- **Decisión que debemos tomar:** si la entidad mantiene pandas, migra a Polars o adopta Polars de forma gradual para generar cada semana el resumen departamental, con un equipo de 8 GB de RAM, un volumen que podría duplicarse en el próximo año (hoy 5,3 millones de filas) y un equipo analista que conoce pandas pero no Polars.
- **Hipótesis inicial:** con la misma entrada Parquet, Polars tardará menos y aumentará menos la memoria que pandas, porque ejecuta la consulta de forma diferida, en streaming y en paralelo, mientras que pandas materializa la tabla antes de agrupar, y esa ventaja se mantendrá si el volumen se duplica. La hipótesis se refutaría si pandas igualara o superara a Polars en algún criterio medido; la celda de la matriz de criterios lo comprueba con una aserción.
- **Resultado equivalente que validamos:** pandas y Polars devolvieron la misma tabla de 1.188 grupos (corte, año, mes y departamento) y sus conteos suman 5.327.501 filas, igual que la comparación previa entre Polars y DuckDB. Usamos las seis mediciones (tres por motor, con un calentamiento previo no contado) y las conservamos todas. Mediana de 1,249 s en pandas (1,237 a 1,252 s) y de 0,144 s en Polars (0,142 a 0,147 s), razón de 8,68 y rangos que no se solapan. Aumento mediano de RSS de 259 MB en pandas y de 22 MB en Polars. La prueba con volumen x2 simulado repite la validación de equivalencia antes de medir y se registra en `benchmark_x2_polars_pandas.csv`.
- **Etapa o recurso que probablemente explica la diferencia:** la ejecución de la consulta, no la lectura de columnas, porque ambos motores leen solo cuatro columnas del Parquet (el plan de Polars muestra `PROJECT 4/6 COLUMNS`). Probablemente pesan la materialización de 5,3 millones de filas de texto en pandas, coherente con sus 259 MB adicionales, y el paralelismo y el streaming de Polars. La celda de diagnóstico mide por separado lectura y agrupación en cada motor e imprime qué etapa domina en pandas; es una sola corrida, así que es una pista y no una prueba.
- **Recomendación para la entidad:** adopción gradual con Polars como destino. Mantener pandas para exploración y procesos actuales, usar Parquet como entrada común, escribir el resumen semanal en Polars y conservar la versión en pandas unas semanas con la misma aserción de equivalencia. Costo de aprendizaje: bajo si se empieza por leer, filtrar, agrupar y contar, con capacitación corta. Memoria: ambos caben en 8 GB en el volumen actual, pero Polars aumenta unas 12 veces menos y deja más margen si el volumen se duplica. Tiempo: 1,25 s frente a 0,14 s son pocos segundos para un proceso semanal, por eso el tiempo no decide por sí solo. Spark no es necesario mientras el volumen quepa en un equipo y no haya uniones grandes ni necesidad de tolerancia a fallos.
- **Límite principal de esta evidencia:** son tres repeticiones por motor, en un único kernel compartido con RSS inicial creciente, con el Parquet ya en caché y en un equipo de 24 hilos lógicos que puede no ser como el de la entidad. El volumen x2 es simulado con una copia del mismo archivo, no datos nuevos, y la medición parte de un Parquet ya preparado y no de los CSV semanales. El costo de aprendizaje y el riesgo de migración no se pueden demostrar con código.
- **Próxima prueba antes de adoptar o escalar:** repetir en el equipo real de 8 GB, con cada motor en un proceso aislado, cinco repeticiones, caché controlada, el proceso completo desde los CSV de la semana y un volumen real mayor (datos de un periodo adicional, no una copia). Allí se verá cómo crecen tiempo y memoria y si Polars mantiene su ventaja con menos núcleos.

# Productos de entrega

Entregue el notebook ejecutado y la carpeta `kit/salidas/pqrs_completos_2023_2025/` con:

- `inventario_fuentes.csv`;
- `control_lectura_polars_duckdb.csv`;
- `equivalencia_polars_duckdb.json`;
- `pqrs_2023_2025_comun.parquet`;
- `benchmark_polars_pandas.csv`;
- `benchmark_polars_pandas_resumen.csv`;
- `equivalencia_polars_pandas.json`;
- `configuracion_benchmark.json`;
- `indicadores_ejercicio_clase.json`;
- un informe breve siguiendo la guía de interpretación.

No es necesario adjuntar los CSV originales, pero el inventario debe permitir reconstruir el ejercicio.
